# Saple role-specific salary moderation-risk demo
This educational notebook uses final moderator decisions to demonstrate an optional Logistic Regression assistant. It never makes a moderation decision; a human remains the final authority.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

ML_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.config import FEATURE_COLUMNS
from src.data_loader import load_dataset, prepare_role_training_data
from src.evaluate import evaluate_classifier
from src.features import build_pipeline
from src.predict import predict_risk
from src.train import split_for_evaluation

## 1. Load reviewed salary data
Generate the fictional demo CSV first if no privacy-safe export is available.

In [ ]:
DATA_PATH = ML_ROOT / 'data' / 'salary_training.csv'
data = load_dataset(DATA_PATH)
print('Dataset shape:', data.shape)
data.head()

## 2. Inspect role counts and enforce the 50-record rule
Only `APPROVED` and `REJECTED` are final training labels. `PENDING` and `FLAGGED` are excluded.

In [ ]:
data.groupby(['role_id', 'role_name']).size().rename('rows')

In [ ]:
ROLE_ID = 1
reviewed, labels, eligibility = prepare_role_training_data(data, ROLE_ID)
eligibility

In [ ]:
if not eligibility['eligible']:
    raise RuntimeError(eligibility['message'])

class_counts = reviewed['moderation_status'].value_counts()
class_counts.plot.bar(title='Final moderator outcomes for selected role')
plt.ylabel('Reviewed submissions')
plt.show()
class_counts

## 3. Split, preprocess, and train
The `Pipeline` median-fills and scales numeric features, fills and one-hot-encodes categorical features, then fits class-balanced Logistic Regression.

In [ ]:
X = reviewed[FEATURE_COLUMNS]
X_train, X_test, y_train, y_test = split_for_evaluation(X, labels)
pipeline = build_pipeline()
pipeline.fit(X_train, y_train)
print(f'Train: {len(X_train)}, test: {len(X_test)}')

## 4. Evaluate
Precision, recall, and F1 deserve more attention than accuracy when suspicious examples are the minority. Synthetic metrics do not represent real-world performance.

In [ ]:
metrics = evaluate_classifier(pipeline, X_test, y_test)
metrics

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, pipeline.predict(X_test), display_labels=['Approved (0)', 'Rejected (1)']
)
plt.title('Salary moderation-risk confusion matrix')
plt.show()

## 5. Score one fictional example
`LOW`, `MEDIUM`, and `HIGH` are demonstration risk bands, never automatic decisions.

In [ ]:
example = {
    'base_salary': 300000,
    'additional_compensation': 0,
    'years_of_experience': 1.5,
    'pay_period': 'MONTHLY',
    'employment_type': 'FULL_TIME',
    'work_mode': 'REMOTE',
    'verification_status': 'UNVERIFIED',
    'salary_year': 2026,
}
artifact = {'pipeline': pipeline, 'roleId': ROLE_ID, 'reviewedSamples': len(reviewed)}
predict_risk(artifact, example)